# 03 - Build Daily Fire Risk Dataset

This notebook merges the grid, daily dates, FIRMS detections, and NDVI into a daily panel dataset. The target predicts whether a grid cell will have at least one fire detection in the next 7 days.

**Temporal leakage rule:** features may use only current and past information. Future fires are used only to create the target, never as input features. Future NDVI and future weather must not be used as features.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Edit this path to the location where you upload the project folder in Google Drive.
PROJECT_DIR = "/content/drive/MyDrive/fire-risk-project"
PROJECT_DIR = Path(PROJECT_DIR)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIRMS_PATH = PROJECT_DIR / "data" / "raw" / "firms" / "firms_viirs_morocco_ws_2023.csv"
GRID_PATH = PROCESSED_DIR / "morocco_ws_grid.geojson"
NDVI_GRID_PATH = PROCESSED_DIR / "ndvi_grid_2023.csv"
OUTPUT_DATASET_PATH = PROCESSED_DIR / "fire_risk_dataset_2023_ndvi_firms.csv"

print("GRID_PATH:", GRID_PATH)
print("FIRMS_PATH:", FIRMS_PATH)
print("NDVI_GRID_PATH:", NDVI_GRID_PATH)
print("OUTPUT_DATASET_PATH:", OUTPUT_DATASET_PATH)


Mounted at /content/drive
GRID_PATH: /content/drive/MyDrive/fire-risk-project/data/processed/morocco_ws_grid.geojson
FIRMS_PATH: /content/drive/MyDrive/fire-risk-project/data/raw/firms/firms_viirs_morocco_ws_2023.csv
NDVI_GRID_PATH: /content/drive/MyDrive/fire-risk-project/data/processed/ndvi_grid_2023.csv
OUTPUT_DATASET_PATH: /content/drive/MyDrive/fire-risk-project/data/processed/fire_risk_dataset_2023_ndvi_firms.csv


In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd


In [3]:
grid = gpd.read_file(GRID_PATH)
grid["grid_id"] = grid["grid_id"].astype(int)
print(grid.shape)
display(grid.head())


(4095, 6)


,grid_id,west,south,east,north,geometry
0,0,-17.20,20.5,-16.95,20.75,"POLYGON ((-16.95 20.5, -16.95 20.75, -17.2 20...."
1,1,-16.95,20.5,-16.70,20.75,"POLYGON ((-16.7 20.5, -16.7 20.75, -16.95 20.7..."
2,2,-16.70,20.5,-16.45,20.75,"POLYGON ((-16.45 20.5, -16.45 20.75, -16.7 20...."
3,3,-16.45,20.5,-16.20,20.75,"POLYGON ((-16.2 20.5, -16.2 20.75, -16.45 20.7..."
4,4,-16.20,20.5,-15.95,20.75,"POLYGON ((-15.95 20.5, -15.95 20.75, -16.2 20...."


In [4]:
firms = pd.read_csv(FIRMS_PATH)
print(firms.shape)
display(firms.head())


(4526, 16)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,request_start_date
0,33.84033,-5.63820,332.41,0.45,0.47,2023-01-01,1256,N,VIIRS,n,2,295.43,2.29,D,0,2023-01-01
1,29.40406,-9.01355,336.19,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.25,3.15,D,0,2023-01-01
2,29.40438,-9.01239,337.86,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.39,4.45,D,0,2023-01-01
3,33.55703,-7.52066,300.68,0.66,0.73,2023-01-02,118,N,VIIRS,n,2,279.15,1.64,N,0,2023-01-01
4,34.59477,-2.34885,298.55,0.34,0.56,2023-01-02,118,N,VIIRS,n,2,277.27,0.80,N,2,2023-01-01


In [5]:
if firms.empty:
    fires_gdf = gpd.GeoDataFrame(columns=list(firms.columns) + ["date", "geometry"], geometry="geometry", crs="EPSG:4326")
else:
    firms["date"] = pd.to_datetime(firms["acq_date"], errors="coerce").dt.normalize()
    firms = firms.dropna(subset=["date", "longitude", "latitude"]).copy()
    fires_gdf = gpd.GeoDataFrame(
        firms,
        geometry=gpd.points_from_xy(firms["longitude"], firms["latitude"]),
        crs="EPSG:4326",
    )

print(fires_gdf.shape)
display(fires_gdf.head())


(4526, 18)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,request_start_date,date,geometry
0,33.84033,-5.63820,332.41,0.45,0.47,2023-01-01,1256,N,VIIRS,n,2,295.43,2.29,D,0,2023-01-01,2023-01-01,POINT (-5.6382 33.84033)
1,29.40406,-9.01355,336.19,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.25,3.15,D,0,2023-01-01,2023-01-01,POINT (-9.01355 29.40406)
2,29.40438,-9.01239,337.86,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.39,4.45,D,0,2023-01-01,2023-01-01,POINT (-9.01239 29.40438)
3,33.55703,-7.52066,300.68,0.66,0.73,2023-01-02,118,N,VIIRS,n,2,279.15,1.64,N,0,2023-01-01,2023-01-02,POINT (-7.52066 33.55703)
4,34.59477,-2.34885,298.55,0.34,0.56,2023-01-02,118,N,VIIRS,n,2,277.27,0.80,N,2,2023-01-01,2023-01-02,POINT (-2.34885 34.59477)


In [6]:
if fires_gdf.empty:
    fires_with_grid = gpd.GeoDataFrame(columns=list(fires_gdf.columns) + ["grid_id"])
else:
    fires_with_grid = gpd.sjoin(
        fires_gdf,
        grid[["grid_id", "geometry"]],
        how="inner",
        predicate="within",
    )

print(fires_with_grid.shape)
display(fires_with_grid.head())


(4526, 20)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,request_start_date,date,geometry,index_right,grid_id
0,33.84033,-5.63820,332.41,0.45,0.47,2023-01-01,1256,N,VIIRS,n,2,295.43,2.29,D,0,2023-01-01,2023-01-01,POINT (-5.6382 33.84033),3491,3491
1,29.40406,-9.01355,336.19,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.25,3.15,D,0,2023-01-01,2023-01-01,POINT (-9.01355 29.40406),2307,2307
2,29.40438,-9.01239,337.86,0.57,0.69,2023-01-01,1436,N,VIIRS,n,2,292.39,4.45,D,0,2023-01-01,2023-01-01,POINT (-9.01239 29.40438),2307,2307
3,33.55703,-7.52066,300.68,0.66,0.73,2023-01-02,118,N,VIIRS,n,2,279.15,1.64,N,0,2023-01-01,2023-01-02,POINT (-7.52066 33.55703),3418,3418
4,34.59477,-2.34885,298.55,0.34,0.56,2023-01-02,118,N,VIIRS,n,2,277.27,0.80,N,2,2023-01-01,2023-01-02,POINT (-2.34885 34.59477),3699,3699


In [7]:
if fires_with_grid.empty:
    fire_daily = pd.DataFrame(columns=["grid_id", "date", "fire_count"])
else:
    fire_daily = (
        fires_with_grid.groupby(["grid_id", "date"])
        .size()
        .reset_index(name="fire_count")
    )

fire_daily["grid_id"] = fire_daily["grid_id"].astype(int)
fire_daily["date"] = pd.to_datetime(fire_daily["date"]).dt.normalize()
print(fire_daily.shape)
display(fire_daily.head())


(2165, 3)


,grid_id,date,fire_count
0,6,2023-03-04,1
1,6,2023-06-21,2
2,6,2023-12-15,1
3,6,2023-12-31,2
4,16,2023-06-18,1


In [8]:
dates = pd.date_range("2023-01-01", "2023-12-31", freq="D")
grid_ids = sorted(grid["grid_id"].unique())

panel = pd.MultiIndex.from_product(
    [grid_ids, dates],
    names=["grid_id", "date"],
).to_frame(index=False)

print(panel.shape)
display(panel.head())


(1494675, 2)


,grid_id,date
0,0,2023-01-01
1,0,2023-01-02
2,0,2023-01-03
3,0,2023-01-04
4,0,2023-01-05


In [9]:
panel = panel.merge(fire_daily, on=["grid_id", "date"], how="left")
panel["fire_count"] = panel["fire_count"].fillna(0).astype(int)
print(panel.shape)
display(panel.head())


(1494675, 3)


,grid_id,date,fire_count
0,0,2023-01-01,0
1,0,2023-01-02,0
2,0,2023-01-03,0
3,0,2023-01-04,0
4,0,2023-01-05,0


In [10]:
ndvi_grid = pd.read_csv(NDVI_GRID_PATH)
ndvi_grid["date"] = pd.to_datetime(ndvi_grid["date"]).dt.normalize()
ndvi_grid["grid_id"] = ndvi_grid["grid_id"].astype(int)
print(ndvi_grid.shape)
display(ndvi_grid.head())


(98280, 4)


,grid_id,date,ndvi_raw_mean,ndvi
0,0,2022-12-19,NaN,NaN
1,0,2023-01-01,NaN,NaN
2,0,2023-01-17,NaN,NaN
3,0,2023-02-02,NaN,NaN
4,0,2023-02-18,NaN,NaN


In [11]:
panel = panel.merge(
    ndvi_grid[["grid_id", "date", "ndvi"]],
    on=["grid_id", "date"],
    how="left",
)
print(panel.shape)
display(panel.head())


(1494675, 4)


,grid_id,date,fire_count,ndvi
0,0,2023-01-01,0,NaN
1,0,2023-01-02,0,NaN
2,0,2023-01-03,0,NaN
3,0,2023-01-04,0,NaN
4,0,2023-01-05,0,NaN


In [12]:
panel = panel.sort_values(["grid_id", "date"]).copy()
panel["ndvi"] = panel.groupby("grid_id")["ndvi"].ffill()
print(panel["ndvi"].isna().mean())
display(panel.head())


0.3178329737233847


,grid_id,date,fire_count,ndvi
0,0,2023-01-01,0,NaN
1,0,2023-01-02,0,NaN
2,0,2023-01-03,0,NaN
3,0,2023-01-04,0,NaN
4,0,2023-01-05,0,NaN


In [13]:
def future_fire_count(series, horizon_days=7):
    future_columns = [series.shift(-offset) for offset in range(1, horizon_days + 1)]
    return pd.concat(future_columns, axis=1).sum(axis=1, min_count=horizon_days)


panel["fire_next_7d_count"] = panel.groupby("grid_id")["fire_count"].transform(
    lambda s: future_fire_count(s, horizon_days=7)
)
panel["fire_count_next_7d"] = panel["fire_next_7d_count"]

panel["fire_risk_label"] = pd.NA
valid_target = panel["fire_next_7d_count"].notna()
panel.loc[valid_target, "fire_risk_label"] = (
    panel.loc[valid_target, "fire_next_7d_count"] > 0
).astype(int)
panel["fire_risk_label"] = panel["fire_risk_label"].astype("Int64")

display(panel[["grid_id", "date", "fire_count", "fire_next_7d_count", "fire_risk_label"]].head(12))


,grid_id,date,fire_count,fire_next_7d_count,fire_risk_label
0,0,2023-01-01,0,0.0,0
1,0,2023-01-02,0,0.0,0
2,0,2023-01-03,0,0.0,0
3,0,2023-01-04,0,0.0,0
4,0,2023-01-05,0,0.0,0
5,0,2023-01-06,0,0.0,0
6,0,2023-01-07,0,0.0,0
7,0,2023-01-08,0,0.0,0
8,0,2023-01-09,0,0.0,0
9,0,2023-01-10,0,0.0,0


In [14]:
panel["month"] = panel["date"].dt.month
panel["dayofyear"] = panel["date"].dt.dayofyear

grouped_fire = panel.groupby("grid_id")["fire_count"]
grouped_ndvi = panel.groupby("grid_id")["ndvi"]

panel["fire_count_lag_1d"] = grouped_fire.shift(1)
panel["fire_count_past_7d"] = grouped_fire.transform(
    lambda s: s.shift(1).rolling(7, min_periods=1).sum()
)
panel["fire_count_past_30d"] = grouped_fire.transform(
    lambda s: s.shift(1).rolling(30, min_periods=1).sum()
)
panel["ndvi_lag_16d"] = grouped_ndvi.shift(16)
panel["ndvi_change_16d"] = panel["ndvi"] - panel["ndvi_lag_16d"]

display(panel.head())


,grid_id,date,fire_count,ndvi,fire_next_7d_count,fire_count_next_7d,fire_risk_label,month,dayofyear,fire_count_lag_1d,fire_count_past_7d,fire_count_past_30d,ndvi_lag_16d,ndvi_change_16d
0,0,2023-01-01,0,NaN,0.0,0.0,0,1,1,NaN,NaN,NaN,NaN,NaN
1,0,2023-01-02,0,NaN,0.0,0.0,0,1,2,0.0,0.0,0.0,NaN,NaN
2,0,2023-01-03,0,NaN,0.0,0.0,0,1,3,0.0,0.0,0.0,NaN,NaN
3,0,2023-01-04,0,NaN,0.0,0.0,0,1,4,0.0,0.0,0.0,NaN,NaN
4,0,2023-01-05,0,NaN,0.0,0.0,0,1,5,0.0,0.0,0.0,NaN,NaN


In [15]:
panel["fire_count_lag_1d"] = panel["fire_count_lag_1d"].fillna(0)
panel["fire_count_past_7d"] = panel["fire_count_past_7d"].fillna(0)
panel["fire_count_past_30d"] = panel["fire_count_past_30d"].fillna(0)
panel["ndvi_lag_16d"] = panel["ndvi_lag_16d"].fillna(panel["ndvi"])
panel["ndvi_change_16d"] = panel["ndvi_change_16d"].fillna(0.0)

dataset = panel.dropna(subset=["ndvi", "fire_next_7d_count", "fire_risk_label"]).copy()
dataset["fire_risk_label"] = dataset["fire_risk_label"].astype(int)

print("Panel rows:", len(panel))
print("Dataset rows after dropping rows without NDVI or complete future target:", len(dataset))
print(dataset["fire_risk_label"].value_counts(dropna=False))
display(dataset.head())


Panel rows: 1494675
Dataset rows after dropping rows without NDVI or complete future target: 1000060
fire_risk_label
0    989926
1     10134
Name: count, dtype: int64


,grid_id,date,fire_count,ndvi,fire_next_7d_count,fire_count_next_7d,fire_risk_label,month,dayofyear,fire_count_lag_1d,fire_count_past_7d,fire_count_past_30d,ndvi_lag_16d,ndvi_change_16d
730,2,2023-01-01,0,0.081758,0.0,0.0,0,1,1,0.0,0.0,0.0,0.081758,0.0
731,2,2023-01-02,0,0.081758,0.0,0.0,0,1,2,0.0,0.0,0.0,0.081758,0.0
732,2,2023-01-03,0,0.081758,0.0,0.0,0,1,3,0.0,0.0,0.0,0.081758,0.0
733,2,2023-01-04,0,0.081758,0.0,0.0,0,1,4,0.0,0.0,0.0,0.081758,0.0
734,2,2023-01-05,0,0.081758,0.0,0.0,0,1,5,0.0,0.0,0.0,0.081758,0.0


In [16]:
OUTPUT_DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)
dataset.to_csv(OUTPUT_DATASET_PATH, index=False)
print(f"Saved daily fire risk dataset to {OUTPUT_DATASET_PATH}")


Saved daily fire risk dataset to /content/drive/MyDrive/fire-risk-project/data/processed/fire_risk_dataset_2023_ndvi_firms.csv
